# 07a · Semantic Segmentation `[local]`

**Goal:** go from boxes to **pixels**: label every pixel with a class. You'll build a **U-Net from scratch**, compute segmentation metrics yourself, compare loss functions and upsampling methods, and score pretrained FCN and DeepLab models on VOC.

| Section | What you'll do |
|---|---|
| 🧠 Intuition | Semantic vs. instance vs. panoptic; encoder–decoder; why skip connections matter |
| 📐 Math | Transposed convolution, atrous (dilated) convolution, IoU/Dice per pixel, Dice loss |
| 🔨 From scratch | Transposed conv by hand, U-Net, pixel IoU/Dice metrics, Dice loss |
| 🚀 Train | U-Net on Oxford Pets (pet vs. background) on your Mac |
| 🚀 Pretrained | FCN vs. DeepLabV3 on VOC 2007, no training |
| 📊 Evaluate | mIoU, Dice, per-class IoU, error maps, boundary vs. interior accuracy |
| 🧪 Experiment | No skip connections, Dice vs. cross-entropy, transposed conv vs. bilinear upsampling |

In [ ]:
# ── Standard first cell ──
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, *pathlib.Path("/kaggle/input").glob("*")]:
    if (_p / "cvlib").is_dir():
        sys.path.insert(0, str(_p)); break
from cvlib.env import setup
env = setup()

import time
import numpy as np
import matplotlib.pyplot as plt
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from PIL import Image
from torchvision import datasets
from cvlib.datasets import cache_pairs, voc2007_segmentation, VOC_CLASSES
from cvlib.loaders import SegGPULoader
from cvlib.metrics import seg_confusion, seg_scores

device = env.device
rng = np.random.default_rng(0)
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def sync():
    if device == "cuda": torch.cuda.synchronize()
    elif device == "mps": torch.mps.synchronize()

---
# 🧠 1 · Intuition

### Three kinds of segmentation

| Task | Output | Example question |
|---|---|---|
| **Semantic** | a class for every pixel | "which pixels are road, car, person?" (two touching people = one "person" region) |
| **Instance** | a separate mask per object | "outline each person separately" (07b) |
| **Panoptic** | both: every pixel gets a class, and countable "things" also get an instance ID | self-driving scene understanding (07b) |

### Encoder–decoder
A classifier **shrinks** the image (224 → 7×7) to understand *what* is there, but loses *where*. Segmentation needs both:
- the **encoder** (any CNN backbone) downsamples to get meaning
- the **decoder** upsamples back to full resolution to place it

### Skip connections (U-Net)
Upsampling a 7×7 map can't magically restore sharp edges: the detail is gone. **U-Net** copies the encoder's high-resolution feature maps across to the decoder at each scale (the horizontal lines in its "U"). The decoder combines *what* (from deep layers) with *where exactly* (from shallow layers). These are different from ResNet's skips (which **add**): U-Net **concatenates**.

### Atrous (dilated) convolution: DeepLab's alternative
Instead of downsampling and then upsampling, keep the resolution higher and use **dilated** convolutions (module 03) to still see a wide context. DeepLab's **ASPP** runs several dilation rates in parallel, giving multi-scale context at one resolution.

### Where it's used
Medical imaging (tumor outlines), satellite land use, self-driving (drivable area), photo editing (background removal), document layout (text regions, module 17), industrial defects.

---
# 📐 2 · The math

### Transposed convolution ("deconvolution")
A stride-2 transposed conv **upsamples** 2×: insert zeros between input pixels, then run an ordinary convolution. Its weights are learned, unlike fixed bilinear interpolation. It's exactly the operation that computes a conv layer's input gradient (module 03), run forward. Output size:

$$H_{out} = (H_{in} - 1)\cdot s - 2p + k$$

With kernel 3 and stride 2, the output pixels get uneven numbers of contributions, which creates **checkerboard artifacts**. Common fixes: kernel divisible by the stride (2×2 or 4×4), or bilinear upsample + 3×3 conv.

### Dilated convolution
A 3×3 kernel with dilation $d$ samples pixels $d$ apart: same 9 weights, but a $(2d+1) \times (2d+1)$ reach. Receptive field grows quickly without losing resolution.

### Per-pixel metrics (from a pixel confusion matrix)
For class $c$ over all pixels in the dataset:

$$\text{IoU}_c = \frac{TP_c}{TP_c + FP_c + FN_c} \qquad \text{Dice}_c = \frac{2\,TP_c}{2\,TP_c + FP_c + FN_c}$$

**mIoU** = mean over classes (the standard benchmark number). Dice is always ≥ IoU, and is popular in medical imaging. **Pixel accuracy** is misleading when the background dominates: predicting "background" everywhere can score 90%.

Important: accumulate TP/FP/FN over the **whole dataset**, then divide. Averaging per-image IoUs gives a different (and noisier) number.

### Losses
- **Pixel-wise cross-entropy:** classification at every pixel, averaged. Pixels labeled *ignore* (255) are skipped.
- **Soft Dice loss:** directly optimizes overlap using probabilities instead of hard decisions:
$$\mathcal{L}_{\text{Dice}} = 1 - \frac{2\sum_i p_i g_i + \epsilon}{\sum_i p_i + \sum_i g_i + \epsilon}$$
  It is insensitive to how many background pixels there are, which helps when the object is tiny. CE + Dice is a common combination.

---
# 🔨 3 · From scratch

### 3.1 · Transposed convolution = zero-insertion + convolution

In [ ]:
x = torch.arange(1.0, 10.0).view(1, 1, 3, 3)
w = torch.randn(1, 1, 3, 3)
ref = F.conv_transpose2d(x, w, stride=2, padding=1, output_padding=1)    # 3×3 → 6×6

z = torch.zeros(1, 1, 5, 5)
z[..., ::2, ::2] = x                                                    # insert zeros between pixels
z = F.pad(z, (1, 2, 1, 2))                                              # pad: k − 1 − p before, + output_padding after
mine = F.conv2d(z, w.flip(-1, -2))                                      # ordinary conv with the flipped kernel
print(f"output {tuple(ref.shape[-2:])} | max diff vs F.conv_transpose2d: {(ref - mine).abs().max():.1e}")

# checkerboard: upsample a constant image with a constant kernel. Ideally the output would be constant too.
flat = torch.ones(1, 1, 8, 8)
ups = {"transposed 3×3, stride 2": F.conv_transpose2d(flat, torch.ones(1, 1, 3, 3), stride=2, padding=1, output_padding=1),
       "transposed 4×4, stride 2": F.conv_transpose2d(flat, torch.ones(1, 1, 4, 4), stride=2, padding=1),
       "bilinear ×2": F.interpolate(flat, scale_factor=2, mode="bilinear", align_corners=False)}
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
for ax, (k, v) in zip(axes, ups.items()):
    ax.imshow(v[0, 0, 2:-2, 2:-2], cmap="viridis"); ax.set_title(k, fontsize=9); ax.axis("off")
plt.suptitle("Upsampling a flat image: uneven overlap → checkerboard"); plt.tight_layout(); plt.show()

With a 3×3 kernel and stride 2, some output pixels receive 4 contributions and others 1 or 2: a built-in checkerboard. A 4×4 kernel (divisible by the stride) overlaps evenly in the interior. Learned weights can partly compensate, but the bias toward the pattern remains.

### 3.2 · U-Net

In [ ]:
def double_conv(c_in, c_out):
    return nn.Sequential(nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
                         nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True))

class UNet(nn.Module):
    def __init__(self, n_classes=2, base=16, depth=4, skips=True, up="transposed"):   # base=16 keeps it fast on a laptop GPU
        super().__init__()
        chs = [base * 2 ** i for i in range(depth + 1)]                   # e.g. 32, 64, 128, 256, 512
        self.downs = nn.ModuleList([double_conv(3, chs[0])] + [double_conv(chs[i], chs[i + 1]) for i in range(depth)])
        self.skips = skips
        self.ups, self.decs = nn.ModuleList(), nn.ModuleList()
        for i in reversed(range(depth)):
            if up == "transposed":
                self.ups.append(nn.ConvTranspose2d(chs[i + 1], chs[i], 2, stride=2))   # learned 2× upsampling
            else:
                self.ups.append(nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), nn.Conv2d(chs[i + 1], chs[i], 3, padding=1)))
            self.decs.append(double_conv(chs[i] * (2 if skips else 1), chs[i]))
        self.head = nn.Conv2d(chs[0], n_classes, 1)                       # 1×1 conv: per-pixel class scores

    def forward(self, x):
        feats = []
        for i, down in enumerate(self.downs):
            x = down(x if i == 0 else F.max_pool2d(x, 2))
            feats.append(x)
        x = feats.pop()                                                   # bottleneck
        for up, dec in zip(self.ups, self.decs):
            x = up(x)
            skip = feats.pop()
            x = dec(torch.cat([x, skip], 1) if self.skips else x)        # U-Net: CONCATENATE encoder features
        return self.head(x)

unet = UNet()
print(f"U-Net: {sum(p.numel() for p in unet.parameters()) / 1e6:.1f}M parameters | output for a 128×128 image: {tuple(unet(torch.zeros(1, 3, 128, 128)).shape)}")

### 3.3 · Soft Dice loss

In [ ]:
def dice_loss(logits, target, ignore_index=255, eps=1.0):
    """Mean soft-Dice loss over classes. logits (B, C, H, W), target (B, H, W)."""
    C = logits.shape[1]
    valid = (target != ignore_index).unsqueeze(1)
    probs = F.softmax(logits, 1) * valid
    onehot = F.one_hot(target.clamp(0, C - 1), C).permute(0, 3, 1, 2) * valid
    inter = (probs * onehot).sum((0, 2, 3))
    denom = probs.sum((0, 2, 3)) + onehot.sum((0, 2, 3))
    return 1 - ((2 * inter + eps) / (denom + eps)).mean()

t = torch.randint(0, 2, (2, 16, 16))
perfect = F.one_hot(t, 2).permute(0, 3, 1, 2).float() * 20
print(f"Dice loss, perfect prediction: {dice_loss(perfect, t):.4f} | random: {dice_loss(torch.randn(2, 2, 16, 16), t):.4f}")

---
# 🚀 4 · Train a U-Net: pets vs. background

Oxford Pets includes a pixel **trimap** for every image: 1 = pet, 2 = background, 3 = uncertain border. We predict 2 classes (pet, background) and mark the border as **ignore** (255): those pixels count neither in the loss nor in the metrics.

In [ ]:
SIZE = 128
pets_tr = datasets.OxfordIIITPet(env.data_dir, split="trainval", target_types="segmentation", download=True)
pets_te = datasets.OxfordIIITPet(env.data_dir, split="test", target_types="segmentation", download=True)
t0 = time.perf_counter()
Xtr, Mtr = cache_pairs("pets_seg_trainval", pets_tr, SIZE)
Xte, Mte = cache_pairs("pets_seg_test", pets_te, SIZE)
print(f"cached in {time.perf_counter() - t0:.0f}s | train {Xtr.shape} | test {Xte.shape}")

def remap(m):
    """trimap 1 = pet → 0, 2 = background → 1, 3 = border → 255 (ignore)"""
    m = np.asarray(m).astype(np.int64)
    return np.where(m == 1, 0, np.where(m == 2, 1, 255)).astype(np.uint8)

Mtr, Mte = remap(Mtr), remap(Mte)
perm = rng.permutation(len(Xtr))
tr_idx, va_idx = np.sort(perm[:3200]), np.sort(perm[3200:])
CLASSES = ["pet", "background"]

def seg_loader(X, M, idx=None, train=False, bs=32):
    idx = np.arange(len(X)) if idx is None else idx
    return SegGPULoader(np.asarray(X[idx]), M[idx], batch_size=bs, shuffle=train, augment=train, mean=MEAN, std=STD, device=device)

palette = np.array([[230, 120, 40], [40, 40, 40]] + [[0, 0, 0]] * 253 + [[255, 255, 255]], np.uint8)
fig, axes = plt.subplots(2, 6, figsize=(13, 4.6))
for k, i in enumerate(rng.choice(len(Xtr), 6, replace=False)):
    axes[0, k].imshow(Xtr[i]); axes[1, k].imshow(palette[Mtr[i]])
    axes[0, k].axis("off"); axes[1, k].axis("off")
plt.suptitle("orange = pet, dark = background, white = ignored border"); plt.tight_layout(); plt.show()

In [ ]:
@torch.no_grad()
def evaluate_seg(model, loader, n_classes=2):
    model.eval(); cm = np.zeros((n_classes, n_classes), np.int64)
    for x, m in loader:
        cm += seg_confusion(model(x).argmax(1).cpu().numpy(), m.cpu().numpy(), n_classes)
    return seg_scores(cm)

def train_seg(model, loss_name="ce", epochs=10, lr=2e-3, idx=tr_idx, verbose=True):
    model = model.to(device)
    tl = seg_loader(Xtr, Mtr, idx, train=True)
    vl = seg_loader(Xtr, Mtr, va_idx)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=epochs * len(tl), pct_start=0.15)
    ce = nn.CrossEntropyLoss(ignore_index=255)
    hist = []
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.perf_counter(); tot = 0
        for x, m in tl:
            out = model(x)
            loss = {"ce": lambda: ce(out, m), "dice": lambda: dice_loss(out, m), "ce+dice": lambda: ce(out, m) + dice_loss(out, m)}[loss_name]()
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            tot += loss.item() / len(tl)
        s = evaluate_seg(model, vl)
        hist.append(s["mIoU"])
        if verbose:
            print(f"epoch {ep:2d} | loss {tot:.4f} | val mIoU {s['mIoU']:.4f} | pet IoU {s['iou'][0]:.4f} | {time.perf_counter() - t0:.0f}s")
    return model, hist

torch.manual_seed(0)
unet, unet_hist = train_seg(UNet(), "ce", epochs=8)

---
# 📊 5 · Evaluate the U-Net

### 5.1 · Test metrics

In [ ]:
test_loader = seg_loader(Xte, Mte)
s = evaluate_seg(unet, test_loader)
print(f"TEST: mIoU {s['mIoU']:.4f} | mean Dice {s['mDice']:.4f} | pixel accuracy {s['pixel_acc']:.4f}")
for c, name in enumerate(CLASSES):
    print(f"  {name:<11} IoU {s['iou'][c]:.4f} | Dice {s['dice'][c]:.4f}")

Note how Dice is always higher than IoU for the same predictions (Dice = 2·IoU / (1 + IoU)). Report which one you use: they aren't interchangeable.

### 5.2 · Predictions and error maps

Error map colors: **green** = correct pet, **red** = false pet (predicted pet on background), **blue** = missed pet, gray = ignored border.

In [ ]:
@torch.no_grad()
def predict_masks(model, X, idx):
    model.eval()
    x, _ = next(iter(SegGPULoader(np.asarray(X[idx]), np.zeros((len(idx), SIZE, SIZE), np.uint8), batch_size=len(idx), mean=MEAN, std=STD, device=device)))
    return model(x).argmax(1).cpu().numpy()

show_idx = rng.choice(len(Xte), 6, replace=False)
pred = predict_masks(unet, Xte, show_idx)
fig, axes = plt.subplots(3, 6, figsize=(13, 6.8))
for k, i in enumerate(show_idx):
    g, p = Mte[i], pred[k]
    err = np.zeros((SIZE, SIZE, 3), np.uint8)
    err[(g == 0) & (p == 0)] = (60, 180, 60); err[(g == 1) & (p == 0)] = (220, 40, 40); err[(g == 0) & (p == 1)] = (40, 80, 230); err[g == 255] = (150, 150, 150)
    cm_i = seg_confusion(p, g, 2)
    axes[0, k].imshow(Xte[i]); axes[1, k].imshow(palette[p]); axes[2, k].imshow(err)
    axes[0, k].set_title(f"pet IoU {seg_scores(cm_i)['iou'][0]:.2f}", fontsize=9)
for a in axes.flat: a.axis("off")
axes[1, 0].set_ylabel("prediction"); plt.suptitle("rows: image | prediction | error map"); plt.tight_layout(); plt.show()

### 5.3 · Where are the errors? Boundary vs. interior

Most segmentation errors sit near object edges. Measure accuracy separately in a thin band around the true boundary and in the interior:

In [ ]:
def boundary_vs_interior(model, X=Xte, M=Mte):
    """Pixel accuracy in a ~2 px band around each pet's outline vs. everywhere else (ignored pixels skipped)."""
    preds = np.concatenate([predict_masks(model, X, np.arange(s_, min(s_ + 256, len(X)))) for s_ in range(0, len(X), 256)])
    band_ok = band_n = in_ok = in_n = 0
    k5 = np.ones((5, 5), np.uint8)
    for g, p in zip(M, preds):
        pet = (g == 0).astype(np.uint8)
        band = (cv2.dilate(pet, k5) - cv2.erode(pet, k5)).astype(bool) & (g != 255)
        inner = (~band) & (g != 255)
        band_ok += (p[band] == g[band]).sum(); band_n += band.sum()
        in_ok += (p[inner] == g[inner]).sum(); in_n += inner.sum()
    return band_ok / band_n, in_ok / in_n, band_n / (band_n + in_n)

b_acc, i_acc, b_frac = boundary_vs_interior(unet)
print(f"pixel accuracy near boundaries: {b_acc:.3f} | in the interior: {i_acc:.3f}")
print(f"boundary band = {b_frac:.1%} of evaluated pixels")

The interior is nearly solved; the boundary is where the remaining error lives, even though it's a small share of pixels. Metrics like **boundary F1** or **boundary IoU** exist to focus on exactly this, and it's what skip connections are for (Experiment 1).

---
# 🚀 6 · Pretrained FCN and DeepLabV3 on VOC, no training

torchvision's segmentation models were trained on a COCO subset using exactly the 20 VOC classes (+ background), so we can score them on VOC 2007's segmentation set directly. FCN and DeepLabV3 here share the same ResNet-50 backbone, so the comparison isolates the **head**: FCN's simple upsampling vs. DeepLab's dilated convolutions + ASPP.

In [ ]:
from torchvision.models import segmentation as seg
voc_items = voc2007_segmentation("val")[:100]                     # 100 of the 213 images: keeps memory and time low on 8 GB Macs
SEG_MODELS = {
    "FCN ResNet-50": (seg.fcn_resnet50, seg.FCN_ResNet50_Weights.DEFAULT),
    "DeepLabV3 ResNet-50": (seg.deeplabv3_resnet50, seg.DeepLabV3_ResNet50_Weights.DEFAULT),
    "DeepLabV3 MobileNetV3": (seg.deeplabv3_mobilenet_v3_large, seg.DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT),
}
voc_res, examples = {}, {}
for name, (fn, weights) in SEG_MODELS.items():
    model = fn(weights=weights).eval().to(device)
    pre = weights.transforms()                                         # resize short side to 520 + normalize
    cm = np.zeros((21, 21), np.int64); times = []
    for k, it in enumerate(voc_items):
        img = Image.open(it["path"]).convert("RGB"); gt = np.array(Image.open(it["mask"]))
        x = pre(img)[None].to(device)
        sync(); t0 = time.perf_counter()
        with torch.no_grad():
            out = model(x)["out"]
        sync(); times.append(time.perf_counter() - t0)
        p = F.interpolate(out, size=gt.shape, mode="bilinear", align_corners=False).argmax(1)[0].cpu().numpy()
        cm += seg_confusion(p, gt, 21)
        if k < 4:
            examples.setdefault(name, []).append(p)
    voc_res[name] = seg_scores(cm) | {"ms": float(np.median(times[3:]) * 1000)}
    print(f"{name:<22} mIoU {voc_res[name]['mIoU']:.4f} | pixel acc {voc_res[name]['pixel_acc']:.4f} | {voc_res[name]['ms']:.0f} ms/img")
    del model
    if device == "mps": torch.mps.empty_cache()                     # free GPU memory before loading the next model

In [ ]:
names21 = ["background"] + VOC_CLASSES
best = max(voc_res, key=lambda k: voc_res[k]["mIoU"])
iou = voc_res[best]["iou"]; order = np.argsort(iou)
plt.figure(figsize=(8, 5.5))
plt.barh([names21[i] for i in order], iou[order]); plt.xlabel("IoU"); plt.title(f"{best}: per-class IoU on VOC 2007 val ({len(voc_items)} images)")
plt.grid(alpha=0.3, axis="x"); plt.tight_layout(); plt.show()

voc_pal = np.zeros((256, 3), np.uint8)
for i in range(21):                                                    # the standard VOC color map
    c, r_, g_, b_ = i, 0, 0, 0
    for j in range(8):
        r_ |= ((c >> 0) & 1) << (7 - j); g_ |= ((c >> 1) & 1) << (7 - j); b_ |= ((c >> 2) & 1) << (7 - j); c >>= 3
    voc_pal[i] = (r_, g_, b_)
voc_pal[255] = (255, 255, 255)
fig, axes = plt.subplots(4, 2 + len(SEG_MODELS), figsize=(15, 11))
for row, it in enumerate(voc_items[:4]):
    axes[row, 0].imshow(Image.open(it["path"])); axes[row, 1].imshow(voc_pal[np.array(Image.open(it["mask"]))])
    for col, name in enumerate(SEG_MODELS, start=2):
        axes[row, col].imshow(voc_pal[examples[name][row]])
for col, t in enumerate(["image", "ground truth"] + list(SEG_MODELS)):
    axes[0, col].set_title(t, fontsize=9)
for a in axes.flat: a.axis("off")
plt.tight_layout(); plt.show()

Background and big, distinctive classes (aeroplane, cat, bus) score high; thin or cluttered ones (**chair**, **bicycle**, **pottedplant**, **diningtable**) score lowest. Bicycles are mostly holes, and chairs are thin and occluded. DeepLab's dilated, multi-scale head typically beats FCN on the same backbone, especially on object boundaries. The white outlines in the ground truth are VOC's *ignore* band.

---
# 🧪 7 · Experiments: predict first, then run

Each run: 6 epochs on 2,000 training images, to stay quick.

In [ ]:
small_idx = tr_idx[:2000]
def quick(model, loss="ce"):
    torch.manual_seed(0)
    m, h = train_seg(model, loss, epochs=6, idx=small_idx, verbose=False)
    return evaluate_seg(m, seg_loader(Xtr, Mtr, va_idx)), m

**Experiment 1: remove the skip connections.** Predict the effect on mIoU, and *where* in the image the damage shows up.

In [ ]:
# Prediction: ...
s_skip, m_skip = quick(UNet(skips=True))
s_noskip, m_noskip = quick(UNet(skips=False))
for name, s_, m_ in [("with skips", s_skip, m_skip), ("without skips", s_noskip, m_noskip)]:
    b_, i_, _ = boundary_vs_interior(m_)
    print(f"{name:<14}: mIoU {s_['mIoU']:.4f} | boundary accuracy {b_:.3f} | interior accuracy {i_:.3f}")
idx3 = show_idx[:3]
p1, p2 = predict_masks(m_skip, Xte, idx3), predict_masks(m_noskip, Xte, idx3)
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for r, i in enumerate(idx3):
    axes[r, 0].imshow(Xte[i]); axes[r, 1].imshow(palette[p1[r]]); axes[r, 2].imshow(palette[p2[r]])
for c, t in enumerate(["image", "U-Net", "no skips"]): axes[0, c].set_title(t)
for a in axes.flat: a.axis("off")
plt.tight_layout(); plt.show()
# mIoU barely moves, because most pixels are interior, and the bottleneck still knows *where the pet roughly is*.
# The damage is at the edges: boundary accuracy drops by several points while interior accuracy is unchanged.
# Without skips the decoder must rebuild outlines from an 8×8 bottleneck: rounded, blobby masks; thin parts
# (ears, tails, legs) suffer most. An average metric can hide exactly the errors an architecture choice causes.

**Experiment 2: the loss function.** Cross-entropy vs. Dice vs. CE + Dice. Predict which gives the best mIoU here, where pets cover a large part of each image.

In [ ]:
# Prediction: ...
for loss in ("ce", "dice", "ce+dice"):
    s_, _ = quick(UNet(), loss)
    print(f"{loss:<8}: mIoU {s_['mIoU']:.4f} | pet IoU {s_['iou'][0]:.4f}")
# With large, balanced objects the differences are small. Dice's advantage shows up when the foreground is
# tiny (lesions in a scan, cracks in a wall), where cross-entropy is swamped by easy background pixels.

**Experiment 3: transposed conv vs. bilinear upsampling in the decoder.** Predict the accuracy difference and whether you'll see checkerboard patterns.

In [ ]:
# Prediction: ...
s_bi, m_bi = quick(UNet(up="bilinear"))
print(f"transposed 2×2: mIoU {s_skip['mIoU']:.4f} | bilinear + conv: mIoU {s_bi['mIoU']:.4f}")
with torch.no_grad():
    x, _ = next(iter(seg_loader(Xte, Mte, show_idx[:1])))
    pp = [F.softmax(m(x), 1)[0, 0].cpu().numpy() for m in (m_skip, m_bi)]
fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
for ax, p_, t in zip(axes, pp, ["P(pet), transposed 2×2", "P(pet), bilinear + conv"]):
    ax.imshow(p_, cmap="magma"); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()
# With a 2×2 kernel and stride 2 there is no uneven overlap, so no checkerboard; both work similarly here.
# The artifacts appear with kernel sizes not divisible by the stride (section 3.1), as in some GAN generators.

---
# 📝 8 · Recap

**Cheat sheet**
- **Semantic** = class per pixel; **instance** = mask per object; **panoptic** = both.
- **Encoder–decoder**: downsample for meaning, upsample for location. **U-Net skips concatenate** high-res encoder features into the decoder for sharp edges.
- Upsampling: **transposed conv** (learned; beware kernel/stride mismatch → checkerboard) or **bilinear + conv**.
- **Dilated convs / ASPP** (DeepLab) widen context without losing resolution.
- Metrics: accumulate a **pixel confusion matrix** over the dataset → **IoU**, **mIoU**, **Dice** (≥ IoU). Pixel accuracy is misleading with dominant backgrounds. Use an **ignore** label for uncertain pixels.
- Losses: pixel **cross-entropy** (with `ignore_index`), **soft Dice** for small foregrounds, or both.
- Errors concentrate at **boundaries**. Thin and cluttered classes are hardest.

**Interview-style questions**
1. Semantic vs. instance vs. panoptic segmentation: give an application for each.
2. Why does U-Net need skip connections? How are they different from ResNet's?
3. What causes checkerboard artifacts in transposed convolutions, and how do you avoid them?
4. How does atrous convolution increase the receptive field without reducing resolution?
5. Why is pixel accuracy a poor metric for segmentation? What do you report instead?
6. Dice vs. IoU: how are they related? When would you train with a Dice loss?
7. Why compute mIoU from a dataset-wide confusion matrix rather than averaging per-image IoU?

**Next:** `07b_instance_segmentation_and_sam.ipynb`: separate masks per object, panoptic quality, and segmenting anything with SAM.